# Phase 2A: EMSCAD Dataset Preparation

This notebook creates a deterministic, cleaned-but-duplicate-preserving EMSCAD dataset for later text-only experiments. It does not generate synthetic scams, train models, create splits, or evaluate classifiers.

## 1. Objective

Prepare the five core recruitment text fields (`title`, `company_profile`, `description`, `requirements`, and `benefits`) while retaining every source record and documenting duplicate and leakage risks.

## 2. Imports and Paths

In [1]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'emscad.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'emscad.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'emscad_clean.csv'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(f'Raw EMSCAD file not found: {RAW_PATH}')

sys.path.insert(0, str(PROJECT_ROOT))
from src.preprocessing import (
    CORE_TEXT_FIELDS,
    analyze_duplicates,
    build_combined_text,
    clean_text,
    create_text_group_id,
)

TARGET_COLUMN = 'fraudulent'
BALANCE_COLUMN = 'in_balanced_dataset'
REPLACEMENT_ARTIFACTS = ('\ufffd', 'ï¿½')
HTML_PATTERN = re.compile(r'<[A-Za-z!/][^>]*>')


## 3. Load Raw EMSCAD

All values are loaded as strings so the source representation is preserved for audit. The raw file is read only and is never altered.

In [2]:
raw_df = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=True)
assert TARGET_COLUMN in raw_df.columns, 'The target column was not found.'
assert set(CORE_TEXT_FIELDS).issubset(raw_df.columns), 'One or more core text fields are missing.'

print(f'Raw shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
print(f'Target values: {sorted(raw_df[TARGET_COLUMN].dropna().unique().tolist())}')
display(raw_df.head(3))

Raw shape: 17,880 rows x 18 columns
Target values: ['f', 't']


,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset
0,Marketing Intern,"US, NY, New York",Marketing,NaN,"<h3>We're Food52, and we've created a groundbr...","<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,f,t,f,Other,Internship,NaN,NaN,Marketing,f,f
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"<h3>90 Seconds, the worlds Cloud Video Product...",<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,<h3><b>What you will get from us</b></h3>\r\n<...,f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,f,f
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,<h3></h3>\r\n<p>Valor Services provides Workfo...,"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,f,t,f,NaN,NaN,NaN,NaN,NaN,f,f


## 4. Duplicate Investigation

Exact duplicate rows and duplicate descriptions are assessed separately. Repeated titles are reported only as a common-title signal: the same title alone does not establish that two advertisements are duplicates.

In [3]:
def redact_for_report(value, limit=280):
    """Keep report examples short and avoid exposing contact details."""
    value = '' if pd.isna(value) else str(value)
    value = re.sub(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', '[EMAIL]', value)
    value = re.sub(r'(?<!\w)(?:\+?\d[\d .()/-]{6,}\d)', '[PHONE]', value)
    return value[:limit] + ('...' if len(value) > limit else '')

exact_group_sizes = raw_df.groupby(list(raw_df.columns), dropna=False).size()
exact_duplicate_groups = exact_group_sizes[exact_group_sizes.gt(1)]
feature_columns = [column for column in raw_df.columns if column != TARGET_COLUMN]
feature_label_counts = raw_df.groupby(feature_columns, dropna=False)[TARGET_COLUMN].nunique()
feature_signature_conflicts = int(feature_label_counts.gt(1).sum())
description_duplicate_stats = analyze_duplicates(raw_df, 'description', TARGET_COLUMN)
title_normalized = raw_df['title'].fillna('').str.replace(r'\s+', ' ', regex=True).str.strip()
title_frame = pd.DataFrame({'normalized_title': title_normalized, TARGET_COLUMN: raw_df[TARGET_COLUMN]})
title_duplicate_stats = analyze_duplicates(title_frame, 'normalized_title', TARGET_COLUMN)

description_groups = raw_df.assign(_description=raw_df['description'].fillna('')).groupby('_description')[TARGET_COLUMN].agg(['size', 'nunique'])
conflicting_descriptions = description_groups.query('size > 1 and nunique > 1').index.tolist()
conflict_example = pd.DataFrame()
if conflicting_descriptions:
    conflict_value = conflicting_descriptions[0]
    conflict_example = raw_df.loc[raw_df['description'].fillna('').eq(conflict_value), ['title', 'description', TARGET_COLUMN]].copy()
    conflict_example.insert(0, 'source_row_id', conflict_example.index)
    conflict_example['description'] = conflict_example['description'].map(redact_for_report)

duplicate_rows = [
    {'section': 'exact_rows', 'metric': 'duplicate rows after first occurrence', 'value': int(raw_df.duplicated().sum()), 'interpretation': 'Rows are retained in emscad_clean.csv.'},
    {'section': 'exact_rows', 'metric': 'exact duplicate groups', 'value': int(len(exact_duplicate_groups)), 'interpretation': 'All exact duplicates include the same target value.'},
    {'section': 'exact_rows', 'metric': 'feature-identical groups with conflicting labels', 'value': feature_signature_conflicts, 'interpretation': 'Features exclude fraudulent; zero supports retaining source records without resolving a direct conflict.'},
    {'section': 'description', 'metric': 'duplicate groups', 'value': description_duplicate_stats['duplicate_groups'], 'interpretation': 'Repeated descriptions can leak across future splits.'},
    {'section': 'description', 'metric': 'duplicate records after first occurrence', 'value': description_duplicate_stats['duplicate_records_after_first'], 'interpretation': 'Includes same-label and conflicting-label groups.'},
    {'section': 'description', 'metric': 'same-label duplicate groups', 'value': description_duplicate_stats['same_label_duplicate_groups'], 'interpretation': 'Use text_group_id to keep these together during future splitting.'},
    {'section': 'description', 'metric': 'conflicting-label groups', 'value': description_duplicate_stats['conflicting_label_groups'], 'interpretation': 'Retain and review; do not silently delete ambiguous source data.'},
    {'section': 'title', 'metric': 'normalized-title duplicate groups', 'value': title_duplicate_stats['duplicate_groups'], 'interpretation': 'Titles alone are not used as an advertisement-duplicate rule.'},
    {'section': 'policy', 'metric': 'processed-data duplicate policy', 'value': 'retain_all_rows', 'interpretation': 'No records are removed. Exact cleaned-text groups receive deterministic identifiers for later group-aware splits.'},
]
duplicate_report = pd.DataFrame(duplicate_rows)
if not conflict_example.empty:
    duplicate_report = pd.concat([
        duplicate_report,
        pd.DataFrame([{
            'section': 'description_example',
            'metric': 'conflicting-label source rows',
            'value': ', '.join(f"{row.source_row_id} ({getattr(row, TARGET_COLUMN)})" for row in conflict_example.itertuples(index=False)),
            'interpretation': 'One repeated raw description occurs with both labels; source rows and labels are shown for review without reproducing the full posting text.'
        }])
    ], ignore_index=True)
duplicate_report.to_csv(TABLES_DIR / 'duplicate_analysis.csv', index=False)
display(duplicate_report)
display(conflict_example)

,section,metric,value,interpretation
0,exact_rows,duplicate rows after first occurrence,235,Rows are retained in emscad_clean.csv.
1,exact_rows,exact duplicate groups,204,All exact duplicates include the same target v...
2,exact_rows,feature-identical groups with conflicting labels,0,Features exclude fraudulent; zero supports ret...
3,description,duplicate groups,1052,Repeated descriptions can leak across future s...
4,description,duplicate records after first occurrence,2785,Includes same-label and conflicting-label groups.
5,description,same-label duplicate groups,1051,Use text_group_id to keep these together durin...
6,description,conflicting-label groups,1,Retain and review; do not silently delete ambi...
7,title,normalized-title duplicate groups,1739,Titles alone are not used as an advertisement-...
8,policy,processed-data duplicate policy,retain_all_rows,No records are removed. Exact cleaned-text gro...
9,description_example,conflicting-label source rows,"9138 (t), 11889 (f), 15236 (f)",One repeated raw description occurs with both ...


,source_row_id,title,description,fraudulent
9138,9138,Front-End/JavaScript Engineer,<p>Workable is a product-driven software compa...,t
11889,11889,Developer (Front-end team),<p>Workable is a product-driven software compa...,f
15236,15236,Developer,<p>Workable is a product-driven software compa...,f


## 5. `in_balanced_dataset` Investigation

In [4]:
if BALANCE_COLUMN not in raw_df.columns:
    raise KeyError(f'{BALANCE_COLUMN} is not present in the raw dataset.')

balance_counts = raw_df[BALANCE_COLUMN].value_counts(dropna=False)
balance_crosstab = pd.crosstab(raw_df[BALANCE_COLUMN], raw_df[TARGET_COLUMN], dropna=False)
balance_rows = []
for balance_value, count in balance_counts.items():
    balance_rows.append({
        'analysis_item': 'value_count',
        'in_balanced_dataset_value': str(balance_value),
        'fraudulent_value': '',
        'count': int(count),
        'percentage_of_dataset': round(100 * count / len(raw_df), 4),
        'interpretation': 'Dataset flag; excluded from all model inputs.'
    })
for balance_value in balance_crosstab.index:
    for target_value in balance_crosstab.columns:
        count = int(balance_crosstab.loc[balance_value, target_value])
        balance_rows.append({
            'analysis_item': 'target_crosstab',
            'in_balanced_dataset_value': str(balance_value),
            'fraudulent_value': str(target_value),
            'count': count,
            'percentage_of_dataset': round(100 * count / len(raw_df), 4),
            'interpretation': 'The t subset has an even 450/450 class split, consistent with a prior balancing/subsetting artifact.'
        })
balance_report = pd.DataFrame(balance_rows)
balance_report.to_csv(TABLES_DIR / 'in_balanced_dataset_analysis.csv', index=False)
display(balance_crosstab)
display(balance_report)

fraudulent,f,t
in_balanced_dataset,,
f,16564,416
t,450,450


,analysis_item,in_balanced_dataset_value,fraudulent_value,count,percentage_of_dataset,interpretation
0,value_count,f,,16980,94.9664,Dataset flag; excluded from all model inputs.
1,value_count,t,,900,5.0336,Dataset flag; excluded from all model inputs.
2,target_crosstab,f,f,16564,92.6398,"The t subset has an even 450/450 class split, ..."
3,target_crosstab,f,t,416,2.3266,"The t subset has an even 450/450 class split, ..."
4,target_crosstab,t,f,450,2.5168,"The t subset has an even 450/450 class split, ..."
5,target_crosstab,t,t,450,2.5168,"The t subset has an even 450/450 class split, ..."


## 6. Data Dictionary and Future Text-Only Scope

In [5]:
BINARY_COLUMNS = {'telecommuting', 'has_company_logo', 'has_questions'}
CATEGORICAL_COLUMNS = {
    'location', 'department', 'salary_range', 'employment_type',
    'required_experience', 'required_education', 'industry', 'function'
}

def column_assessment(column):
    if column == TARGET_COLUMN:
        return 'target', 'No', 'Outcome label; must never be a model input.'
    if column == BALANCE_COLUMN:
        return 'dataset artifact', 'No', 'Likely prior balancing/subsetting flag; high leakage risk.'
    if column in CORE_TEXT_FIELDS:
        return 'text', 'Yes', 'Core recruitment text used in the Phase 2A text-only representation.'
    if column in BINARY_COLUMNS:
        return 'binary metadata', 'No', 'Potential shortcut feature; excluded from the primary text-only model.'
    if column == 'salary_range':
        return 'categorical metadata', 'No', 'Sparse range-like metadata; retain raw value only for later review.'
    if column in CATEGORICAL_COLUMNS:
        return 'categorical metadata', 'No', 'Contextual metadata; excluded to preserve the initial text-only scope.'
    return 'metadata', 'No', 'Not part of the defined primary text-only representation.'

dictionary_rows = []
for column in raw_df.columns:
    non_null = raw_df[column].dropna()
    examples = ' | '.join(redact_for_report(value, limit=80) for value in non_null.head(3))
    category, future_text_model, reason = column_assessment(column)
    missing_count = int(raw_df[column].isna().sum())
    dictionary_rows.append({
        'column_name': column,
        'original_dtype': str(raw_df[column].dtype),
        'unique_values': int(raw_df[column].nunique(dropna=True)),
        'missing_count': missing_count,
        'missing_percentage': round(100 * missing_count / len(raw_df), 4),
        'example_values': examples,
        'category': category,
        'consider_for_future_text_only_model': future_text_model,
        'reason': reason,
    })
data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary.to_csv(TABLES_DIR / 'data_dictionary.csv', index=False)
display(data_dictionary)

,column_name,original_dtype,unique_values,missing_count,missing_percentage,example_values,category,consider_for_future_text_only_model,reason
0,title,object,11231,0,0.0000,Marketing Intern | Customer Service - Cloud Vi...,text,Yes,Core recruitment text used in the Phase 2A tex...
1,location,object,3105,346,1.9351,"US, NY, New York | NZ, , Auckland | US, IA, Wever",categorical metadata,No,Contextual metadata; excluded to preserve the ...
2,department,object,1337,11547,64.5805,Marketing | Success | Sales,categorical metadata,No,Contextual metadata; excluded to preserve the ...
3,salary_range,object,874,15012,83.9597,[PHONE] | [PHONE] | [PHONE],categorical metadata,No,Sparse range-like metadata; retain raw value o...
4,company_profile,object,1710,3308,18.5011,"<h3>We're Food52, and we've created a groundbr...",text,Yes,Core recruitment text used in the Phase 2A tex...
5,description,object,15095,0,0.0000,"<p>Food52, a fast-growing, James Beard Award-w...",text,Yes,Core recruitment text used in the Phase 2A tex...
6,requirements,object,12119,2689,15.0391,<ul>\r\n<li>Experience with content management...,text,Yes,Core recruitment text used in the Phase 2A tex...
7,benefits,object,6510,7196,40.2461,<h3><b>What you will get from us</b></h3>\r\n<...,text,Yes,Core recruitment text used in the Phase 2A tex...
8,telecommuting,object,2,0,0.0000,f | f | f,binary metadata,No,Potential shortcut feature; excluded from the ...
9,has_company_logo,object,2,0,0.0000,t | t | t,binary metadata,No,Potential shortcut feature; excluded from the ...


## 7. HTML Cleaning and Replacement-Character Handling

The reusable `clean_text` function decodes HTML entities, removes HTML tags, converts known replacement artifacts to spaces, and collapses whitespace. It deliberately preserves URLs, email-like strings, phone numbers, digits, case, and normal punctuation.

In [6]:
cleaning_tests = pd.DataFrame([
    {'case': 'null', 'raw': None, 'expected': ''},
    {'case': 'html_and_entity', 'raw': '<p>Alpha &amp; Beta<br>Gamma&nbsp;</p>', 'expected': 'Alpha & Beta Gamma'},
    {'case': 'url_preserved', 'raw': '<p>Apply: https://example.org/a?x=1&amp;y=2</p>', 'expected': 'Apply: https://example.org/a?x=1&y=2'},
    {'case': 'replacement_placeholder', 'raw': 'Role\ufffdName', 'expected': 'Role Name'},
])
cleaning_tests['cleaned'] = cleaning_tests['raw'].map(clean_text)
cleaning_tests['passed'] = cleaning_tests['cleaned'].eq(cleaning_tests['expected'])
assert cleaning_tests['passed'].all(), 'A text cleaning check failed.'
display(cleaning_tests)

def contains_html(series):
    return series.fillna('').astype(str).str.contains(HTML_PATTERN, na=False)

def contains_replacement_artifact(series):
    text = series.fillna('').astype(str)
    return text.str.contains(REPLACEMENT_ARTIFACTS[0], regex=False) | text.str.contains(REPLACEMENT_ARTIFACTS[1], regex=False)

raw_html_records = pd.concat([contains_html(raw_df[field]) for field in CORE_TEXT_FIELDS], axis=1).any(axis=1)
raw_replacement_records = pd.concat([contains_replacement_artifact(raw_df[field]) for field in raw_df.columns], axis=1).any(axis=1)
replacement_rows = []
for column in raw_df.columns:
    affected = int(contains_replacement_artifact(raw_df[column]).sum())
    if affected:
        replacement_rows.append({
            'column': column,
            'affected_records_before_cleaning': affected,
            'handling': 'Known replacement placeholders are replaced with a single space; surrounding content is retained.'
        })
replacement_report = pd.DataFrame(replacement_rows)
replacement_report.to_csv(TABLES_DIR / 'replacement_character_analysis.csv', index=False)
display(replacement_report)

,case,raw,expected,cleaned,passed
0,null,None,,,True
1,html_and_entity,<p>Alpha &amp; Beta<br>Gamma&nbsp;</p>,Alpha & Beta Gamma,Alpha & Beta Gamma,True
2,url_preserved,<p>Apply: https://example.org/a?x=1&amp;y=2</p>,Apply: https://example.org/a?x=1&y=2,Apply: https://example.org/a?x=1&y=2,True
3,replacement_placeholder,Role�Name,Role Name,Role Name,True


,column,affected_records_before_cleaning,handling
0,description,6,Known replacement placeholders are replaced wi...
1,requirements,19,Known replacement placeholders are replaced wi...
2,benefits,1,Known replacement placeholders are replaced wi...


## 8. Missing-Value Handling and Canonical Text Construction

Only the cleaned representation maps missing core text values to empty strings. Raw metadata is not imputed or changed. The canonical `text` field includes only non-empty core fields, with explicit field labels.

In [7]:
cleaned_fields = pd.DataFrame(index=raw_df.index)
for field in CORE_TEXT_FIELDS:
    cleaned_fields[field] = raw_df[field].map(clean_text)

processed_df = cleaned_fields.copy()
processed_df.insert(0, 'source_row_id', raw_df.index.astype(int))
processed_df['text'] = processed_df.apply(build_combined_text, axis=1)
processed_df['fraudulent'] = raw_df[TARGET_COLUMN].astype(str)
processed_df['text_group_id'] = processed_df['text'].map(create_text_group_id)

assert len(processed_df) == len(raw_df), 'Preprocessing must preserve all source rows.'
assert processed_df['text_group_id'].nunique() <= len(processed_df)
display(processed_df.head(3))

,source_row_id,title,company_profile,description,requirements,benefits,text,fraudulent,text_group_id
0,0,Marketing Intern,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,,Title: Marketing Intern\n\nCompany Profile: We...,f,text_455ed01e319c9ea9b5c5c0643caa838e9c54a21ad...
1,1,Customer Service - Cloud Video Production,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome! Do y...,What we expect from you: \n Your key responsi...,What you will get from us \n Through being pa...,Title: Customer Service - Cloud Video Producti...,f,text_e53853f3e5f14f620c2454e029263f3d9e8d56674...
2,2,Commissioning Machinery Assistant (CMA),Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,,Title: Commissioning Machinery Assistant (CMA)...,f,text_c2c46ba83f17edd98d243cf2b8d4d5df38bb53fdd...


## 9. Duplicate/Group Identifiers

`text_group_id` is a SHA-256 hash of the exact cleaned canonical text. It is a provenance and future split-control field, not a model feature. Rows in the same exact-text group should be assigned to the same future train/validation/test split.

In [8]:
processed_group_labels = processed_df.groupby('text_group_id')['fraudulent'].nunique()
processed_duplicate_groups = processed_df['text_group_id'].value_counts()
processed_duplicate_groups = processed_duplicate_groups[processed_duplicate_groups.gt(1)]
print(f'Exact cleaned-text duplicate groups: {len(processed_duplicate_groups):,}')
print(f'Cleaned-text groups with conflicting labels: {int(processed_group_labels.gt(1).sum()):,}')
print('No duplicates are removed in this Phase 2A dataset.')

Exact cleaned-text duplicate groups: 683
Cleaned-text groups with conflicting labels: 0
No duplicates are removed in this Phase 2A dataset.


## 10. Quality-Control Checks and Before/After Comparison

In [9]:
processed_html_records = pd.concat([contains_html(processed_df[field]) for field in CORE_TEXT_FIELDS], axis=1).any(axis=1)
processed_replacement_records = pd.concat([contains_replacement_artifact(processed_df[field]) for field in CORE_TEXT_FIELDS], axis=1).any(axis=1)
raw_core_missing_cells = int(raw_df[CORE_TEXT_FIELDS].isna().sum().sum())
processed_core_empty_cells = int(processed_df[CORE_TEXT_FIELDS].eq('').sum().sum())
raw_core_whitespace_cells = int(raw_df[CORE_TEXT_FIELDS].apply(lambda column: column.notna() & column.str.fullmatch(r'\s*', na=False)).sum().sum())
processed_empty_text_records = int(processed_df['text'].eq('').sum())
raw_all_core_empty_records = int(raw_df[CORE_TEXT_FIELDS].fillna('').apply(lambda column: column.str.strip().eq('')).all(axis=1).sum())
cleaned_exact_rows_after_first = int(processed_df[CORE_TEXT_FIELDS + ['fraudulent']].duplicated().sum())

before_after = pd.DataFrame([
    {'metric': 'row_count', 'before': len(raw_df), 'after': len(processed_df), 'note': 'All source records retained.'},
    {'metric': 'legitimate_records (fraudulent=f)', 'before': int(raw_df[TARGET_COLUMN].eq('f').sum()), 'after': int(processed_df['fraudulent'].eq('f').sum()), 'note': 'Target values unchanged.'},
    {'metric': 'fraudulent_records (fraudulent=t)', 'before': int(raw_df[TARGET_COLUMN].eq('t').sum()), 'after': int(processed_df['fraudulent'].eq('t').sum()), 'note': 'Target values unchanged.'},
    {'metric': 'missing core-text cells', 'before': raw_core_missing_cells, 'after': 0, 'note': 'Only processed core text uses empty strings for raw missing values.'},
    {'metric': 'empty or whitespace-only core-text cells', 'before': raw_core_whitespace_cells, 'after': processed_core_empty_cells, 'note': 'After includes raw missing values represented as empty strings.'},
    {'metric': 'records with all core fields empty', 'before': raw_all_core_empty_records, 'after': processed_empty_text_records, 'note': 'Canonical text omits empty field sections.'},
    {'metric': 'records containing HTML in core fields', 'before': int(raw_html_records.sum()), 'after': int(processed_html_records.sum()), 'note': 'HTML markup removed from cleaned core fields.'},
    {'metric': 'records containing replacement artifacts', 'before': int(raw_replacement_records.sum()), 'after': int(processed_replacement_records.sum()), 'note': 'Post-cleaning count is measured in the cleaned core fields.'},
    {'metric': 'exact duplicate rows after first', 'before': int(raw_df.duplicated().sum()), 'after': cleaned_exact_rows_after_first, 'note': 'After compares cleaned core fields plus target; no rows are removed.'},
    {'metric': 'exact cleaned-text duplicate groups', 'before': np.nan, 'after': int(len(processed_duplicate_groups)), 'note': 'Tracked with text_group_id for future group-aware splitting.'},
])
before_after.to_csv(TABLES_DIR / 'preprocessing_before_after.csv', index=False)
display(before_after)

,metric,before,after,note
0,row_count,17880.0,17880,All source records retained.
1,legitimate_records (fraudulent=f),17014.0,17014,Target values unchanged.
2,fraudulent_records (fraudulent=t),866.0,866,Target values unchanged.
3,missing core-text cells,13193.0,0,Only processed core text uses empty strings fo...
4,empty or whitespace-only core-text cells,0.0,13220,After includes raw missing values represented ...
5,records with all core fields empty,0.0,0,Canonical text omits empty field sections.
6,records containing HTML in core fields,17863.0,0,HTML markup removed from cleaned core fields.
7,records containing replacement artifacts,24.0,0,Post-cleaning count is measured in the cleaned...
8,exact duplicate rows after first,235.0,1823,After compares cleaned core fields plus target...
9,exact cleaned-text duplicate groups,NaN,683,Tracked with text_group_id for future group-aw...


## 11. Before/After Examples

Examples are intentionally shortened and redact email addresses and phone-like strings in this report only. The processed dataset itself preserves the cleaned natural-language content.

In [10]:
def first_index(mask, fallback=0):
    matches = raw_df.index[mask]
    return int(matches[0]) if len(matches) else fallback

example_specs = [
    ('html_markup', first_index(raw_html_records), 'HTML tags and entities are removed while readable text is retained.'),
    ('missing_field', first_index(raw_df['company_profile'].isna()), 'A missing core field becomes an empty processed field; no content is fabricated.'),
    ('replacement_character', first_index(raw_replacement_records), 'Known replacement placeholders become spaces; surrounding content remains.'),
    ('multiple_core_fields', first_index(raw_df[CORE_TEXT_FIELDS].notna().all(axis=1)), 'Multiple cleaned fields are combined using explicit field labels.'),
    ('normal_record', first_index((~raw_html_records) & (~raw_replacement_records)), 'A record without detected HTML or replacement artifacts.'),
]
example_rows = []
for example_type, source_row_id, note in example_specs:
    raw_combined = '\n\n'.join(
        f'{field}: {raw_df.at[source_row_id, field] if pd.notna(raw_df.at[source_row_id, field]) else ""}'
        for field in CORE_TEXT_FIELDS
    )
    example_rows.append({
        'example_type': example_type,
        'source_row_id': source_row_id,
        'fraudulent': raw_df.at[source_row_id, TARGET_COLUMN],
        'raw_text_excerpt': redact_for_report(raw_combined, limit=700),
        'cleaned_text_excerpt': redact_for_report(processed_df.at[source_row_id, 'text'], limit=700),
        'notes': note,
    })
cleaning_examples = pd.DataFrame(example_rows).drop_duplicates(subset=['example_type'])
cleaning_examples.to_csv(TABLES_DIR / 'cleaning_examples.csv', index=False)
display(cleaning_examples)

,example_type,source_row_id,fraudulent,raw_text_excerpt,cleaned_text_excerpt,notes
0,html_markup,0,f,title: Marketing Intern\n\ncompany_profile: <h...,Title: Marketing Intern\n\nCompany Profile: We...,HTML tags and entities are removed while reada...
1,missing_field,5,f,title: Accounting Clerk\n\ncompany_profile: \n...,Title: Accounting Clerk\n\nDescription: Job Ov...,A missing core field becomes an empty processe...
2,replacement_character,1041,f,title: E Commerce Business Analyst \n\ncompany...,Title: E Commerce Business Analyst\n\nDescript...,Known replacement placeholders become spaces; ...
3,multiple_core_fields,1,f,title: Customer Service - Cloud Video Producti...,Title: Customer Service - Cloud Video Producti...,Multiple cleaned fields are combined using exp...
4,normal_record,1365,f,title: STNA/ CNA \n\ncompany_profile: \n\ndesc...,Title: STNA/ CNA\n\nDescription: Helping Hands...,A record without detected HTML or replacement ...


## 12. Save the Processed Base Dataset

In [11]:
processed_df.to_csv(PROCESSED_PATH, index=False, encoding='utf-8')
print(f'Saved {len(processed_df):,} duplicate-preserving records to: {PROCESSED_PATH}')
print('source_row_id is a generated 0-based raw-row position, not an original EMSCAD identifier.')

Saved 17,880 duplicate-preserving records to: C:\Users\Rajesh Singh\Desktop\Fraud Detection\AI_Job_Fraud_Detection\data\processed\emscad_clean.csv
source_row_id is a generated 0-based raw-row position, not an original EMSCAD identifier.


## 13. Phase 2A Findings

- The raw source remains untouched and every row is preserved in `emscad_clean.csv`.
- The primary representation is text only: labeled, cleaned combinations of the five core recruitment fields.
- `fraudulent`, `in_balanced_dataset`, and all metadata are excluded from canonical text.
- `text_group_id` must be retained for future group-aware splitting; it is not a model feature.
- One duplicate-description group has conflicting target labels, so duplicate deletion requires a later, explicit methodological decision.
- No model training, AI scam generation, dataset splitting, or evaluation is performed in this notebook.